# 彩铅画风 LoRA 训练（新版 `sd-scripts` / Google Colab）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YYYYYHHHHHH/lora_train_test/blob/main/lesson19/LoRA_train_pencil_drawing_sd_scripts_Colab.ipynb)

这是 `LoRA_train_pencil_drawing.ipynb` 的迁移版，保留原训练目标：

- 使用 Anything V5（SD 1.5 系）作为基础模型；
- 下载原来的 `caiqian_style` 彩铅数据集；
- 通过 BLIP 生成图片描述；
- 训练 rank 32、alpha 16 的标准 LoRA；
- 同时训练 UNet 和 Text Encoder，每个 epoch 保存权重并生成样图；
- 最后使用 `sd-scripts` 加载 LoRA 做推理验证。

主要兼容性调整：

- 使用仍在维护的 `kohya-ss/sd-scripts`，固定到 `v0.11.1`；
- 不再安装已经与 Colab 不兼容的 `torch==2.0.0+cu118`、`xformers==0.0.19`；
- 沿用 Colab 自带 PyTorch，使用原生 SDPA；
- 不再调用已经过时的 `finetune/make_captions.py`，改用 Hugging Face BLIP；
- 默认把结果保存到 Google Drive，避免 Colab 断线后丢失。

请使用 **GPU 运行时**，建议 T4、L4、A100 或更高。第一次运行请按顺序执行所有单元格。


## 1. 检查 Colab GPU

如果下面报错或没有显示 NVIDIA GPU，请在 Colab 中选择：`运行时 → 更改运行时类型 → T4 GPU`。


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=True)

import torch

assert torch.cuda.is_available(), "没有检测到 CUDA GPU，请先给 Colab 启用 GPU 运行时。"
gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {gpu_name} ({gpu_vram_gb:.1f} GB VRAM)")


## 2. 用户配置

默认参数对应原 Notebook。若使用自己的图片或底模，只需修改这一格。

- `USE_GOOGLE_DRIVE=True`：权重、样图和日志会保存到云端硬盘。
- `TRIGGER_WORD`：新加入的风格触发词。推理时把它放在 prompt 开头。
- 如果 T4 显存不足，把 `TRAIN_BATCH_SIZE` 从 6 改成 2，并把 `GRADIENT_CHECKPOINTING` 改为 `True`。


In [ ]:
# 仓库与资源
SD_SCRIPTS_REF = "v0.11.1"
SD_SCRIPTS_URL = "https://github.com/kohya-ss/sd-scripts.git"
DATASET_URL = "https://huggingface.co/datasets/litmonster0521/pencildrawing/resolve/main/caiqian_style.tar"
BASE_MODEL_URL = "https://civitai.com/api/download/models/90854"

# 训练含义
PROJECT_NAME = "painting"
TRIGGER_WORD = "caiqian_style"
RESOLUTION = 512
DATASET_REPEATS = 1
NUM_EPOCHS = 10
TRAIN_BATCH_SIZE = 6
NETWORK_DIM = 32
NETWORK_ALPHA = 16
UNET_LR = 1e-4
TEXT_ENCODER_LR = 5e-5
CLIP_SKIP = 2
MIXED_PRECISION = "fp16"
GRADIENT_CHECKPOINTING = False
SEED = 42

# 文件位置
USE_GOOGLE_DRIVE = True
WORK_ROOT = Path("/content/lora_pencil_sd_scripts")
REPO_DIR = Path("/content/sd-scripts")
RAW_DATA_DIR = WORK_ROOT / "raw_data"
TRAIN_DATA_DIR = WORK_ROOT / "train_data"
CONFIG_DIR = WORK_ROOT / "config"
MODEL_DIR = WORK_ROOT / "pretrained_model"
BASE_MODEL_PATH = MODEL_DIR / "anything_v5.safetensors"

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/LoRA/painting")
else:
    OUTPUT_DIR = WORK_ROOT / "output" / PROJECT_NAME

SAMPLE_DIR = OUTPUT_DIR / "sample"
LOGGING_DIR = OUTPUT_DIR / "logs"
INFERENCE_DIR = OUTPUT_DIR / "inference"

for directory in [WORK_ROOT, RAW_DATA_DIR, TRAIN_DATA_DIR, CONFIG_DIR, MODEL_DIR, OUTPUT_DIR, SAMPLE_DIR, LOGGING_DIR, INFERENCE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("工作目录：", WORK_ROOT)
print("训练输出：", OUTPUT_DIR)


## 3. 安装固定版本的 `sd-scripts`

这一格不会更换 Colab 自带的 PyTorch，也不会安装 xformers。训练时使用 `--sdpa`，因此避开旧 Notebook 中的版本冲突。

请尽量从一个全新的 Colab 运行时开始运行。如果你在同一运行时里安装过其他版本的 `transformers`、`diffusers` 或 `accelerate`，建议先重启运行时。


In [ ]:
if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", SD_SCRIPTS_REF, SD_SCRIPTS_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "fetch", "--depth", "1", "origin", f"refs/tags/{SD_SCRIPTS_REF}:refs/tags/{SD_SCRIPTS_REF}"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", SD_SCRIPTS_REF], cwd=REPO_DIR, check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "-r", "requirements.txt"],
    cwd=REPO_DIR,
    check=True,
)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"sd-scripts {SD_SCRIPTS_REF} 安装完成，commit={commit}")
print("使用 Colab 自带 PyTorch：", torch.__version__)


## 4. 下载并检查数据集和基础模型

这里继续使用原项目中的彩铅数据集和 Anything V5 下载地址。下载支持断点续传，并会检查模型是否确实是 Safetensors 文件，而不是错误页面。


In [ ]:
import tarfile


def download_with_wget(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["wget", "-c", url, "-O", str(destination)], check=True)
    assert destination.exists() and destination.stat().st_size > 0, f"下载失败：{destination}"


archive_path = WORK_ROOT / "caiqian_style.tar"
extracted_dir = RAW_DATA_DIR / "caiqian_style"
if not extracted_dir.exists():
    # 每次在尚未解压时调用 wget -c；这样 Colab 中断留下的半截压缩包也会自动续传。
    download_with_wget(DATASET_URL, archive_path)
    with tarfile.open(archive_path) as archive:
        try:
            archive.extractall(RAW_DATA_DIR, filter="data")
        except TypeError:  # 兼容较旧 Python
            archive.extractall(RAW_DATA_DIR)

image_extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
source_images = sorted(path for path in extracted_dir.rglob("*") if path.suffix.lower() in image_extensions)
assert source_images, f"没有在 {extracted_dir} 找到图片，请检查数据集下载。"

for source_path in source_images:
    destination = TRAIN_DATA_DIR / source_path.name
    if not destination.exists():
        shutil.copy2(source_path, destination)

from safetensors import safe_open

model_is_valid = False
if BASE_MODEL_PATH.exists():
    try:
        with safe_open(BASE_MODEL_PATH, framework="pt", device="cpu") as model_file:
            model_is_valid = len(model_file.keys()) > 100
    except Exception:
        model_is_valid = False

if not model_is_valid:
    # wget -c 会续传未完成的模型文件。
    download_with_wget(BASE_MODEL_URL, BASE_MODEL_PATH)

with safe_open(BASE_MODEL_PATH, framework="pt", device="cpu") as model_file:
    model_key_count = len(model_file.keys())
assert model_key_count > 100, "基础模型文件不像有效的 Stable Diffusion Safetensors，请检查 CivitAI 下载地址。"

train_images = sorted(path for path in TRAIN_DATA_DIR.iterdir() if path.suffix.lower() in image_extensions)
print(f"训练图片：{len(train_images)} 张")
print(f"基础模型：{BASE_MODEL_PATH} ({BASE_MODEL_PATH.stat().st_size / 1024**3:.2f} GB, {model_key_count} tensors)")


In [ ]:
# 预览部分训练图片
import math

import matplotlib.pyplot as plt
from PIL import Image

preview_paths = train_images[: min(6, len(train_images))]
columns = 3
rows = math.ceil(len(preview_paths) / columns)
fig, axes = plt.subplots(rows, columns, figsize=(12, 4 * rows))
axes = list(getattr(axes, "flat", [axes]))

for axis, path in zip(axes, preview_paths):
    axis.imshow(Image.open(path).convert("RGB"))
    axis.set_title(path.name)
    axis.axis("off")
for axis in axes[len(preview_paths):]:
    axis.axis("off")
plt.tight_layout()


## 5. 使用新版 BLIP 生成描述

当前 `sd-scripts` 文档已经说明旧 `finetune/make_captions.py` 不再推荐使用，因此这里直接使用 `Salesforce/blip-image-captioning-base`。

每个描述都会写成同名 `.txt` 文件，并在开头加入固定风格触发词，例如：

```text
caiqian_style, a woman wearing a hat and holding a flower
```

如果要重新生成全部描述，把 `OVERWRITE_CAPTIONS` 改成 `True`。


In [ ]:
OVERWRITE_CAPTIONS = False
CAPTION_BATCH_SIZE = 4
BLIP_MODEL_ID = "Salesforce/blip-image-captioning-base"

caption_targets = [path for path in train_images if OVERWRITE_CAPTIONS or not path.with_suffix(".txt").exists()]

if caption_targets:
    from transformers import BlipForConditionalGeneration, BlipProcessor

    processor = BlipProcessor.from_pretrained(BLIP_MODEL_ID)
    caption_model = BlipForConditionalGeneration.from_pretrained(
        BLIP_MODEL_ID,
        torch_dtype=torch.float16,
    ).to("cuda")
    caption_model.eval()

    for start in range(0, len(caption_targets), CAPTION_BATCH_SIZE):
        batch_paths = caption_targets[start : start + CAPTION_BATCH_SIZE]
        batch_images = [Image.open(path).convert("RGB") for path in batch_paths]
        inputs = processor(images=batch_images, return_tensors="pt")
        pixel_values = inputs["pixel_values"].to("cuda", dtype=torch.float16)

        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
            generated_ids = caption_model.generate(
                pixel_values=pixel_values,
                max_new_tokens=50,
                num_beams=3,
            )

        captions = processor.batch_decode(generated_ids, skip_special_tokens=True)
        for image_path, caption in zip(batch_paths, captions):
            normalized_caption = " ".join(caption.strip().split())
            image_path.with_suffix(".txt").write_text(
                f"{TRIGGER_WORD}, {normalized_caption}\n",
                encoding="utf-8",
            )
        print(f"已生成 {min(start + CAPTION_BATCH_SIZE, len(caption_targets))}/{len(caption_targets)}")

    del caption_model, processor, inputs, pixel_values, generated_ids
    torch.cuda.empty_cache()
else:
    print("所有图片已经有 .txt 描述，跳过 BLIP。")

caption_files = sorted(TRAIN_DATA_DIR.glob("*.txt"))
assert len(caption_files) == len(train_images), "图片和描述文件数量不一致。"
for caption_path in caption_files[:10]:
    print(caption_path.name, "=>", caption_path.read_text(encoding="utf-8").strip())


## 6. 创建新版数据集配置和采样 Prompt

仍然使用 512 分辨率、宽高比分桶和一个 epoch 一次采样。由于开启 latent cache 后翻转增强不会生效，这个迁移版把 `flip_aug` 明确设为 `false`，与旧 Notebook 的实际训练行为一致。


In [ ]:
import toml

dataset_config = {
    "general": {
        "enable_bucket": True,
        "caption_extension": ".txt",
        "shuffle_caption": True,
        "keep_tokens": 1,
        "bucket_reso_steps": 64,
        "bucket_no_upscale": False,
    },
    "datasets": [
        {
            "resolution": RESOLUTION,
            "min_bucket_reso": 256,
            "max_bucket_reso": 1024,
            "flip_aug": False,
            "color_aug": False,
            "subsets": [
                {
                    "image_dir": str(TRAIN_DATA_DIR),
                    "class_tokens": TRIGGER_WORD,
                    "num_repeats": DATASET_REPEATS,
                }
            ],
        }
    ],
}

DATASET_CONFIG_PATH = CONFIG_DIR / "dataset_config.toml"
DATASET_CONFIG_PATH.write_text(toml.dumps(dataset_config), encoding="utf-8")

negative_prompt = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, blurry"
sample_lines = [
    f"{TRIGGER_WORD}, a beautiful girl, colored pencil drawing --n {negative_prompt} --w 512 --h 512 --d 1024 --l 7 --s 28",
    f"{TRIGGER_WORD}, a cat sitting beside a window, colored pencil drawing --n {negative_prompt} --w 512 --h 512 --d 2048 --l 7 --s 28",
]
SAMPLE_PROMPT_PATH = CONFIG_DIR / "sample_prompts.txt"
SAMPLE_PROMPT_PATH.write_text("\n".join(sample_lines) + "\n", encoding="utf-8")

estimated_steps = math.ceil(len(train_images) * DATASET_REPEATS / TRAIN_BATCH_SIZE) * NUM_EPOCHS
print(DATASET_CONFIG_PATH.read_text(encoding="utf-8"))
print("采样 Prompt：")
print(SAMPLE_PROMPT_PATH.read_text(encoding="utf-8"))
print(f"预计训练步数约为 {estimated_steps}。")


## 7. 训练 LoRA

默认值对应旧 Notebook：rank/alpha `32/16`、UNet 学习率 `1e-4`、文本编码器学习率 `5e-5`、AdamW8bit、10 epochs、batch size 6、clip skip 2。

若出现 CUDA out of memory：回到“用户配置”单元格，将 `TRAIN_BATCH_SIZE=2`、`GRADIENT_CHECKPOINTING=True`，然后从创建配置处重新运行。


In [ ]:
assert BASE_MODEL_PATH.exists(), "基础模型不存在。"
assert len(train_images) == len(caption_files) > 0, "训练图片或描述不完整。"

train_command = [
    "accelerate",
    "launch",
    "--num_cpu_threads_per_process=2",
    f"--mixed_precision={MIXED_PRECISION}",
    str(REPO_DIR / "train_network.py"),
    f"--pretrained_model_name_or_path={BASE_MODEL_PATH}",
    f"--dataset_config={DATASET_CONFIG_PATH}",
    f"--output_dir={OUTPUT_DIR}",
    f"--output_name={PROJECT_NAME}",
    "--save_model_as=safetensors",
    "--network_module=networks.lora",
    f"--network_dim={NETWORK_DIM}",
    f"--network_alpha={NETWORK_ALPHA}",
    f"--unet_lr={UNET_LR}",
    f"--text_encoder_lr={TEXT_ENCODER_LR}",
    "--optimizer_type=AdamW8bit",
    "--lr_scheduler=constant",
    "--lr_warmup_steps=0",
    f"--train_batch_size={TRAIN_BATCH_SIZE}",
    f"--max_train_epochs={NUM_EPOCHS}",
    "--gradient_accumulation_steps=1",
    f"--mixed_precision={MIXED_PRECISION}",
    f"--save_precision={MIXED_PRECISION}",
    "--save_every_n_epochs=1",
    "--sample_every_n_epochs=1",
    f"--sample_prompts={SAMPLE_PROMPT_PATH}",
    "--sample_sampler=euler_a",
    f"--clip_skip={CLIP_SKIP}",
    "--max_token_length=225",
    "--cache_latents",
    "--vae_batch_size=4",
    "--sdpa",
    "--max_data_loader_n_workers=2",
    f"--seed={SEED}",
    f"--logging_dir={LOGGING_DIR}",
    f"--log_prefix={PROJECT_NAME}",
    "--log_with=tensorboard",
]
if GRADIENT_CHECKPOINTING:
    train_command.append("--gradient_checkpointing")

print("即将执行：")
print(" \\\n+  ".join(train_command))
subprocess.run(train_command, cwd=REPO_DIR, check=True)

FINAL_LORA_PATH = OUTPUT_DIR / f"{PROJECT_NAME}.safetensors"
assert FINAL_LORA_PATH.exists(), f"没有找到最终 LoRA：{FINAL_LORA_PATH}"
print("训练完成：", FINAL_LORA_PATH)


## 8. 查看训练过程样图


In [ ]:
sample_paths = sorted(
    path for path in SAMPLE_DIR.glob("*") if path.suffix.lower() in image_extensions
)
print(f"找到 {len(sample_paths)} 张训练样图：{SAMPLE_DIR}")

if sample_paths:
    shown_paths = sample_paths[-min(8, len(sample_paths)):]
    columns = 4
    rows = math.ceil(len(shown_paths) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(16, 4 * rows))
    axes = list(getattr(axes, "flat", [axes]))
    for axis, path in zip(axes, shown_paths):
        axis.imshow(Image.open(path).convert("RGB"))
        axis.set_title(path.name)
        axis.axis("off")
    for axis in axes[len(shown_paths):]:
        axis.axis("off")
    plt.tight_layout()
else:
    print("没有找到样图，请先完成训练。")


## 9. 使用训练好的 LoRA 做推理

推理仍使用 Anything V5 底模和 `sd-scripts`，避免格式转换。修改 `INFERENCE_PROMPT` 后可重复运行。


In [ ]:
INFERENCE_PROMPT = f"{TRIGGER_WORD}, a girl wearing sunglasses and earrings, colored pencil drawing"
NEGATIVE_PROMPT = "lowres, bad anatomy, bad hands, text, error, missing fingers, extra digits, cropped, worst quality, low quality, jpeg artifacts, signature, watermark, blurry"

FINAL_LORA_PATH = OUTPUT_DIR / f"{PROJECT_NAME}.safetensors"
assert FINAL_LORA_PATH.exists(), "请先完成训练，或把 FINAL_LORA_PATH 改成某个 epoch 的 .safetensors 文件。"

inference_command = [
    sys.executable,
    str(REPO_DIR / "gen_img.py"),
    f"--ckpt={BASE_MODEL_PATH}",
    f"--outdir={INFERENCE_DIR}",
    "--network_module=networks.lora",
    "--network_weights",
    str(FINAL_LORA_PATH),
    "--network_mul",
    "1.0",
    "--sdpa",
    "--fp16",
    "--no_preview",
    f"--prompt={INFERENCE_PROMPT} --n {NEGATIVE_PROMPT}",
    "--W=512",
    "--H=512",
    "--seed=1024",
    "--scale=7",
    "--sampler=euler_a",
    "--steps=20",
    "--images_per_prompt=4",
    "--batch_size=4",
    f"--clip_skip={CLIP_SKIP}",
]

subprocess.run(inference_command, cwd=REPO_DIR, check=True)
print("推理图片目录：", INFERENCE_DIR)


In [ ]:
generated_paths = sorted(
    path for path in INFERENCE_DIR.glob("*") if path.suffix.lower() in image_extensions
)
shown_paths = generated_paths[-4:]
assert shown_paths, "没有找到推理图片。"

fig, axes = plt.subplots(1, len(shown_paths), figsize=(4 * len(shown_paths), 4))
axes = list(getattr(axes, "flat", [axes]))
for axis, path in zip(axes, shown_paths):
    axis.imshow(Image.open(path).convert("RGB"))
    axis.set_title(path.name)
    axis.axis("off")
plt.tight_layout()


## 输出说明

- 最终 LoRA：`painting.safetensors`
- 每轮权重：`painting-000001.safetensors` 等
- 训练样图：`sample/`
- TensorBoard 日志：`logs/`
- 推理结果：`inference/`

默认情况下这些文件都位于 Google Drive 的：

```text
MyDrive/LoRA/painting/
```

在 A1111、Forge 或 ComfyUI 中加载 `painting.safetensors` 后，建议在正向提示词开头加入 `caiqian_style`。
